# Jersey digit detector (YOLO26s)

Trains a detector for single digits 0-9 on player crops, on the zip built by `scripts/build_jersey_dataset.py`.
It replaces the capstone model, which was trained on mislabelled classes (balls, whole players and whole-number
boxes merged in as the digits 0-2) and with horizontal flips (a mirrored digit is a different digit, and a
mirrored two-digit number reads backwards). Details: `docs/results/actions.md`.

- Data: the two Roboflow sets with digit classes, split **by match**: test = Japan-Poland, Thailand and
  Poland-Italy (teams in no training match), valid = two other matches, train = the rest.
- No horizontal flips (`fliplr=0`).

1. Runtime -> Change runtime type -> **T4 GPU**
2. **Run all** cells. The best weights are copied to `MyDrive/volleyball/models/jersey_digits_yolo26s.pt`.
3. If Colab disconnects, refresh the page and **Run all** again: training resumes from `last.pt` on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
!pip -q install ultralytics
import ultralytics; ultralytics.checks()


In [ ]:
# Dataset zip copied from the laptop to MyDrive/volleyball/datasets/
DATASET_ZIP = '/content/drive/MyDrive/volleyball/datasets/jersey_digits.zip'
!rm -rf /content/data && mkdir -p /content/data
!unzip -q "$DATASET_ZIP" -d /content/data
!find /content/data -maxdepth 2 -name data.yaml


In [ ]:
import glob, yaml, os
cfg = sorted(glob.glob('/content/data/**/data.yaml', recursive=True))[0]
root = os.path.dirname(cfg)
d = yaml.safe_load(open(cfg))
d['path'] = root
for split, name in (('train', 'train'), ('val', 'valid'), ('test', 'test')):
    d[split] = os.path.join(root, name, 'images')
yaml.safe_dump(d, open(cfg, 'w'))
print(cfg); print(d)


In [ ]:
from ultralytics import YOLO
import torch
# Checkpoints go to Drive so a Colab disconnect does not lose training; rerunning resumes.
RUNS = '/content/drive/MyDrive/volleyball/runs'
RUN_NAME = 'jersey_digits_v1'
last = f'{RUNS}/{RUN_NAME}/weights/last.pt'
# last.pt has epoch == -1 once training finished (or stopped early): then skip, don't retrain.
finished = os.path.exists(last) and torch.load(last, map_location='cpu', weights_only=False).get('epoch') == -1

if finished:
    print('training already finished:', last)
elif os.path.exists(last):
    print('resuming from', last)
    model = YOLO(last)
    model.train(resume=True)
else:
    # crops are 92-184 px wide, so 320 px input is enough
    model = YOLO('yolo26s.pt')
    model.train(data=cfg, epochs=150, imgsz=320, batch=64, patience=40,
                fliplr=0.0,           # a mirrored digit is a different digit
                degrees=5, mosaic=0.5,
                project=RUNS, name=RUN_NAME, exist_ok=True, save_period=10)


In [ ]:
model = YOLO(f'{RUNS}/{RUN_NAME}/weights/best.pt')
# test = matches whose teams are in no training match
m = model.val(data=cfg, split='test', imgsz=320)
print(f'test mAP50 {m.box.map50:.3f}  mAP50-95 {m.box.map:.3f}  precision {m.box.mp:.3f}  recall {m.box.mr:.3f}')
for i, c in enumerate(m.box.ap_class_index):
    print(f'  digit {int(c)}: AP50 {m.box.ap50[i]:.3f}')
import shutil
dst = '/content/drive/MyDrive/volleyball/models'
os.makedirs(dst, exist_ok=True)
shutil.copy(f'{RUNS}/{RUN_NAME}/weights/best.pt', f'{dst}/jersey_digits_yolo26s.pt')
shutil.copy(f'{RUNS}/{RUN_NAME}/results.png', f'{dst}/jersey_digits_v1_results.png')
print('saved to', dst)
